# Hierarchical Sparse Coding — Demo

**Hierarchical Sparse Coding (HSC)** は、階層的な辞書 $D_1, D_2, \dots, D_L$ を学習するスパースコーディングの枠組みです。
各層は前の層の表現をさらに精緻化し、LISTA 風のニューラルエンコーダによる高速推論もサポートします。

| ソルバ | 説明 |
|--------|------|
| ISTA | 固定辞書に対する逐次プロキシ勾配法 |
| MFISTA | モノトニック FISTA — より速い収束保証 |
| LISTA | 辞書を初期値とする学習済みエンコーダ（1パス） |
| Hybrid | LISTA で初期化 → ISTA で精緻化（主手法） |
| Hybrid+MFISTA | LISTA で初期化 → MFISTA で精緻化 |

**本デモでは seed=1 のみを使用** し、CPU 上で MNIST に全手法を学習します。
再現性を確保するため、すべての乱数シードを最初に設定しています。

In [4]:
# ============================================================
# 0. Setup — imports and seed
# ============================================================
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import random
import numpy as np
import torch
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Reproducibility — seed 1 only
SEED = 1
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Add project root to path
import sys
_DEMO_DIR = Path.cwd()
sys.path.insert(0, str(_DEMO_DIR))

from HierarchicalSparseCoding import (
    Config,
    UnifiedHSCInference,
    compute_hsc_layer_steps,
    hierarchical_losses,
    normalize_dictionary_,
    sparsity_stats,
    save_dictionary_grid,
    compute_effective_dictionaries,
)
from train_hsc import (
    make_dataset_loaders,
    prepare_eval_batch,
    build_eval_inference_context,
    collect_validation_snapshot,
)

# Plotting style
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
})

# Color scheme matching experiment_plotters.py
MODE_COLORS = {
    "ista": "#56B4E9",
    "mfista": "#E69F00",
    "lista": "#009E73",
    "hybrid": "#0072B2",
    "hybrid_mfista": "#CC79A7",
}
MODE_MARKERS = {
    "ista": "o", "mfista": "s", "lista": "^",
    "hybrid": "D", "hybrid_mfista": "P",
}

print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
print(f"PyTorch version: {torch.__version__}")

Device: cpu
PyTorch version: 2.8.0


---

# 1. Train all methods

5つのソルバをすべて MNIST で学習します。アーキテクチャは共通:
- **2層**: dims `[256, 64]`
- **Lambda** = 0.05（各層）
- **Beta** = 1.0（結合重み）
- **25エポック**, バッチサイズ 256

これは実験 E1 (core comparison) に相当します。

In [5]:
# ============================================================
# 1. Train all methods
# ============================================================
from train_hsc import train_main

MODES = ["ista", "mfista", "lista", "hybrid", "hybrid_mfista"]
EPOCHS = 25
BATCH_SIZE = 256
LAYER_DIMS = [256, 64]
LAMBDAS = [0.05, 0.05]
BETAS = [1.0]

# Method-specific inference steps (matching run_experiments.sh defaults)
METHOD_STEPS = {
    "ista": {"infer_steps": 50, "lista_steps": 1},
    "mfista": {"infer_steps": 20, "lista_steps": 1},
    "lista": {"infer_steps": 50, "lista_steps": 1},
    "hybrid": {"infer_steps": 5, "lista_steps": 1},
    "hybrid_mfista": {"infer_steps": 5, "lista_steps": 1},
}

HYBRID_PRETRAIN = 5

results = {}

for mode in MODES:
    print(f"\n{'='*60}")
    print(f"Training: {mode} (seed={SEED})")
    print(f"{'='*60}")

    steps = METHOD_STEPS[mode]
    cfg = Config(
        mode=mode,
        lista_variant="shared",
        layer_dims=LAYER_DIMS,
        lambdas=LAMBDAS,
        betas=BETAS,
        lr_D=1e-3,
        lr_E=1e-3,
        infer_steps=steps["infer_steps"],
        lista_steps=steps["lista_steps"],
        eta_scale=1.0,
        hybrid_pretrain_epochs=HYBRID_PRETRAIN if mode == "hybrid" else 0,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        dc_center=False,
        print_every=500,
        device="cpu",
    )

    artifacts = train_main(cfg, dataset_name="mnist", seed=SEED)
    results[mode] = {
        "Ds": artifacts.Ds,
        "infer_module": artifacts.infer_module,
        "run_dir": artifacts.run_dir,
    }

print("\nAll training complete.")


Training: ista (seed=1)
save_dir = data/raw_data/hsc_ista_shared_20260624_160210
mode            ista | phase            ista | ep 01 step 000000 | energy 38.6639 | rec_x 34.5666 | rec_h 2.1269 | sparse 1.9704 | a1: |.|_1 0.1227, active 0.8603 | a2: |.|_1 0.1248, active 0.6650
mode            ista | phase            ista | ep 03 step 000500 | energy 6.0839 | rec_x 2.4477 | rec_h 0.5794 | sparse 3.0568 | a1: |.|_1 0.1735, active 0.8847 | a2: |.|_1 0.2613, active 0.8208
mode            ista | phase            ista | ep 05 step 001000 | energy 5.7999 | rec_x 2.3635 | rec_h 0.5853 | sparse 2.8511 | a1: |.|_1 0.1618, active 0.8618 | a2: |.|_1 0.2436, active 0.7940
mode            ista | phase            ista | ep 08 step 001500 | energy 5.5507 | rec_x 2.2488 | rec_h 0.5566 | sparse 2.7453 | a1: |.|_1 0.1558, active 0.8410 | a2: |.|_1 0.2349, active 0.7681
mode            ista | phase            ista | ep 10 step 002000 | energy 5.5712 | rec_x 2.2930 | rec_h 0.5567 | sparse 2.7215 | a1: |.|

---

# 2. Learned Dictionaries

各手法が学習した辞書原子（atoms）を可視化します。
- **Layer 1 ($D_1$)**: 生の特徴（エッジ、テクスチャなど）— 画像空間 `[784, K]` に直接描画可能
- **Effective ($D_1 \cdot D_2$)**: 入力を直接再構成する合成辞書 — $D_1$ と $D_2$ の積で得られる入力空間の原子
- **右カラム**: Layer 2 ($D_2$) は潜在空間 `[K_prev, K_curr]` にあるため画像描画不可。代わりに原子値の散布図を表示


In [6]:
# ============================================================
# 2. Dictionary visualization
# ============================================================

H, W = 28, 28  # MNIST image size

fig, axes = plt.subplots(len(MODES), 3, figsize=(9, len(MODES) * 3.5))

for row_idx, mode in enumerate(MODES):
    Ds = results[mode]["Ds"]

    # Layer 1 dictionary (input space: [784, K]) - visualizable as images
    save_dictionary_grid(Ds[0], H, W, f"/tmp/dict_l1_{mode}.png", n_show=64)
    ax = axes[row_idx][0]
    ax.imshow(plt.imread(f"/tmp/dict_l1_{mode}.png"))
    ax.axis("off")
    if row_idx == 0:
        ax.set_title("Layer 1 (D1)", fontsize=12)
    else:
        ax.set_title(f"{mode} - Layer 1", fontsize=9)

    # Effective dictionary D_eff^(2) = D1*D2 (input space: [784, K'])
    eff_dicts = compute_effective_dictionaries([d.detach() for d in Ds])

    # Middle column: Effective dictionary (D1*D2) - input-space atoms
    save_dictionary_grid(eff_dicts[1], H, W, f"/tmp/dict_eff_{mode}.png", n_show=64)
    ax = axes[row_idx][1]
    ax.imshow(plt.imread(f"/tmp/dict_eff_{mode}.png"))
    ax.axis("off")
    if row_idx == 0:
        ax.set_title("Effective (D1*D2)", fontsize=12)

    # Right column: D2 summary - atoms are normalized so all norms = 1.0.
    # Show shape and statistics as text instead of trying to visualize.
    D2 = Ds[1].detach().cpu()
    stats_text = (f"{mode} - Layer 2 Dictionary\n"
                  f"Shape: {tuple(D2.shape)}\n"
                  "L2 norm (all): 1.0000\n"
                  f"Mean value: {D2.mean():.4f}\n"
                  f"Std value: {D2.std():.4f}\n"
                  f"Min/Max: {D2.min():.3f} / {D2.max():.3f}")
    ax.text(0.5, 0.5, stats_text, ha="center", va="center", fontsize=10,
            transform=ax.transAxes, family='monospace')
    ax.axis("off")

fig.suptitle("Learned Dictionaries by Method", y=0.98, fontsize=16)
plt.tight_layout()
plt.savefig(str(_DEMO_DIR / "demo_dict_comparison.png"), dpi=150, bbox_inches="tight")
print(f"Saved: demo_dict_comparison.png")


Saved: demo_dict_comparison.png


---

# 3. Reconstruction Examples

各手法による MNIST 画像の再構成結果を比較します。
1行目: オリジナル入力、2行目以降: 各層の有効辞書 $D_1 \cdots D_l$ で再構成した像。

In [8]:
# ============================================================
# 3. Reconstruction examples
# ============================================================

from HierarchicalSparseCoding import show_hsc_recon_examples_nlayer

# Load a small test batch for visualization
_, _, val_loader, test_loader = make_dataset_loaders("mnist", batch_size=8)
test_images, _ = next(iter(test_loader))

all_recon_paths = []

for mode in MODES:
    Ds = results[mode]["Ds"]
    infer_module = results[mode]["infer_module"]
    
    x = prepare_eval_batch(Config(), test_images)
    Ds_for_inf, lambdas, betas, etas = build_eval_inference_context(
        Config(), [d.detach() for d in Ds]
    )
    
    codes = infer_module(
        x=x,
        Ds_for_inference=Ds_for_inf,
        lambdas=lambdas,
        betas=betas,
        infer_steps=METHOD_STEPS[mode]["infer_steps"],
        eta_scale=1.0,
        etas=etas,
    )
    
    eff_dicts = compute_effective_dictionaries([d.detach() for d in Ds])
    
    save_path = f"/tmp/recon_{mode}.png"
    show_hsc_recon_examples_nlayer(
        x=x, codes=codes, eff_dicts=eff_dicts,
        H=H, W=W, n=8,
        save_path=save_path,
        title=f"{mode} (seed={SEED})",
    )
    all_recon_paths.append(save_path)

# Display side by side
fig, axes = plt.subplots(1, len(MODES), figsize=(4 * len(MODES), 5))
for idx, (mode, path) in enumerate(zip(MODES, all_recon_paths)):
    axes[idx].imshow(plt.imread(path))
    axes[idx].axis("off")
    axes[idx].set_title(mode, fontsize=12)

fig.suptitle("Reconstruction Examples (seed=1)", fontsize=14, y=0.98)
plt.tight_layout()
plt.savefig(str(_DEMO_DIR / "demo_recon.png"), dpi=150, bbox_inches="tight")
print(f"Saved: demo_recon.png")

TypeError: show_hsc_recon_examples_nlayer() got an unexpected keyword argument 'save_path'

---

# 4. Performance Comparison

全手法のテストセットにおける最終評価指標を比較します。

In [ ]:
# ============================================================
# 4. Performance comparison
# ============================================================

# Collect test metrics from each trained model
metrics = {}
for mode in MODES:
    cfg = Config(
        mode=mode,
        layer_dims=LAYER_DIMS,
        lambdas=LAMBDAS,
        betas=BETAS,
        infer_steps=METHOD_STEPS[mode]["infer_steps"],
        lista_steps=METHOD_STEPS[mode]["lista_steps"],
        batch_size=BATCH_SIZE,
        device="cpu",
    )
    _, _, val_loader, test_loader = make_dataset_loaders("mnist", batch_size=BATCH_SIZE)
    
    row = collect_validation_snapshot(
        cfg=cfg,
        Ds=results[mode]["Ds"],
        infer_module=results[mode]["infer_module"],
        val_loader=test_loader,
        run_name=mode,
        epoch=EPOCHS,
        run_dir=Path("/tmp/demo_metrics"),
        split_name="test",
    )
    metrics[mode] = {
        "loss": row["loss"],
        "rec_x": row["rec_x"],
        "sparse": row["sparse"],
        "infer_time_ms": row["infer_time_ms"],
    }

# Print table
print("\nTest Metrics (seed=1)")
print("-" * 65)
print(f'{"Method":<12} {"Loss":>8} {"Rec X":>10} {"Sparse":>10} {"Infer (ms)":>12}')
print("-" * 65)
for mode in MODES:
    m = metrics[mode]
    print(f'{mode:<12} {m["loss"]:8.4f} {m["rec_x"]:10.4f} {m["sparse"]:10.4f} {m["infer_time_ms"]:12.3f}')
print("-" * 65)

In [ ]:
# ============================================================
# 4b. Bar chart of final metrics
# ============================================================

import pandas as pd

df_metrics = pd.DataFrame(metrics).T

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

# Loss bar chart
colors = [MODE_COLORS[m] for m in MODES]
axes[0].bar(MODES, df_metrics["loss"], color=colors)
axes[0].set_title("Final Test Loss")
axes[0].set_ylabel("Loss")
axes[0].tick_params(axis="x", rotation=15)

# Inference time bar chart
axes[1].bar(MODES, df_metrics["infer_time_ms"], color=colors)
axes[1].set_title("Final Inference Time")
axes[1].set_ylabel("Time (ms/batch)")
axes[1].tick_params(axis="x", rotation=15)

fig.suptitle(f"Performance Comparison (seed={SEED})", fontsize=14, y=0.98)
plt.tight_layout()
plt.savefig(str(_DEMO_DIR / "demo_metrics.png"), dpi=150, bbox_inches="tight")
print(f"Saved: demo_metrics.png")

---

# 5. Learning Curves

訓練中の loss, reconstruction error, sparsity penalty の遷移を比較します。

In [ ]:
# ============================================================
# 5. Learning curves from saved CSVs
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for col_idx, (metric, title) in enumerate([
    ("loss", "Training Loss"),
    ("rec_x", "Reconstruction Error"),
    ("sparse", "Sparsity Penalty"),
]):
    ax = axes[col_idx]
    for mode in MODES:
        run_dir = results[mode]["run_dir"]
        csv_path = run_dir / "train_log.csv"
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            ax.plot(
                df["epoch"],
                df[metric],
                marker=MODE_MARKERS.get(mode, "o"),
                linewidth=2,
                color=MODE_COLORS.get(mode),
                label=mode,
            )
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.25)

# Shared legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(MODES), frameon=False,
           bbox_to_anchor=(0.5, 0.96))

fig.suptitle(f"Learning Curves (seed={SEED})", fontsize=14)
plt.tight_layout(rect=[0, 0.08, 1, 0.94])
plt.savefig(str(_DEMO_DIR / "demo_learning_curves.png"), dpi=150, bbox_inches="tight")
print(f"Saved: demo_learning_curves.png")

---

# 6. Sparsity Analysis

各層でのコードのスパース性（L1ノルム平均・活性化割合）を手法ごとに比較します。

In [ ]:
# ============================================================
# 6. Sparsity analysis
# ============================================================

# Load a batch for sparsity measurement
_, _, val_loader, test_loader = make_dataset_loaders("mnist", batch_size=256)
test_images, _ = next(iter(test_loader))
x = prepare_eval_batch(Config(), test_images)

sparsity_data = {"layer": [], "method": [], "metric": [], "value": []}

for mode in MODES:
    Ds = results[mode]["Ds"]
    infer_module = results[mode]["infer_module"]
    
    Ds_for_inf, lambdas, betas, etas = build_eval_inference_context(
        Config(), [d.detach() for d in Ds]
    )
    codes = infer_module(
        x=x,
        Ds_for_inference=Ds_for_inf,
        lambdas=lambdas,
        betas=betas,
        infer_steps=METHOD_STEPS[mode]["infer_steps"],
        eta_scale=1.0,
        etas=etas,
    )
    
    for level, a in enumerate(codes):
        stats = sparsity_stats(a)
        sparsity_data["layer"].extend([level + 1] * 2)
        sparsity_data["method"].extend([mode] * 2)
        sparsity_data["metric"].extend(["L1 Mean", "Active Fraction"])
        sparsity_data["value"].extend([stats["l1_mean"], stats["active_frac"]])

sp_df = pd.DataFrame(sparsity_data)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, metric_name in enumerate(["L1 Mean", "Active Fraction"]):
    ax = axes[ax_idx]
    subset = sp_df[sp_df["metric"] == metric_name]
    
    for mode in MODES:
        mdf = subset[subset["method"] == mode].sort_values("layer")
        ax.plot(
            mdf["layer"],
            mdf["value"],
            marker=MODE_MARKERS.get(mode, "o"),
            linewidth=2,
            color=MODE_COLORS.get(mode),
            label=mode,
        )
    
    ax.set_title(metric_name)
    ax.set_xlabel("Layer")
    ax.grid(True, alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(MODES), frameon=False,
           bbox_to_anchor=(0.5, 0.96))

fig.suptitle(f"Sparsity by Layer and Method (seed={SEED})", fontsize=14)
plt.tight_layout(rect=[0, 0.08, 1, 0.94])
plt.savefig(str(_DEMO_DIR / "demo_sparsity.png"), dpi=150, bbox_inches="tight")
print(f"Saved: demo_sparsity.png")

---

# 7. Depth Scaling

Hybrid 手法において、層数（1, 2, 4）を変えたときの性能変化を調べます。
これは実験 E3 (depth scaling) に相当します。

In [ ]:
# ============================================================
# 7. Depth scaling (Hybrid method)
# ============================================================

DEPTHS = {1: [256], 2: [256, 64], 4: [256, 192, 128, 64]}
depth_results = {}

for depth, dims in DEPTHS.items():
    lambdas_d = [0.05] * depth
    betas_d = [1.0] * (depth - 1) if depth > 1 else []
    
    print(f"\nTraining Hybrid with {depth} layers: dims={dims}")
    cfg = Config(
        mode="hybrid",
        lista_variant="shared",
        layer_dims=dims,
        lambdas=lambdas_d,
        betas=betas_d,
        lr_D=1e-3,
        lr_E=1e-3,
        infer_steps=5,
        lista_steps=1,
        eta_scale=1.0,
        hybrid_pretrain_epochs=HYBRID_PRETRAIN,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        dc_center=False,
        print_every=500,
        device="cpu",
    )
    
    artifacts = train_main(cfg, dataset_name="mnist", seed=SEED)
    depth_results[depth] = {
        "Ds": artifacts.Ds,
        "infer_module": artifacts.infer_module,
    }

# Compare metrics across depths
depth_metrics = []
for depth, data in depth_results.items():
    cfg = Config(
        mode="hybrid",
        layer_dims=DEPTHS[depth],
        lambdas=[0.05] * depth,
        betas=[1.0] * (depth - 1) if depth > 1 else [],
        infer_steps=5,
        lista_steps=1,
        batch_size=BATCH_SIZE,
        device="cpu",
    )
    _, _, val_loader, test_loader = make_dataset_loaders("mnist", batch_size=BATCH_SIZE)
    row = collect_validation_snapshot(
        cfg=cfg,
        Ds=data["Ds"],
        infer_module=data["infer_module"],
        val_loader=test_loader,
        run_name=f"hybrid_L{depth}",
        epoch=EPOCHS,
        run_dir=Path("/tmp/demo_depth"),
        split_name="test",
    )
    depth_metrics.append({
        "layers": depth,
        "loss": row["loss"],
        "rec_x": row["rec_x"],
        "sparse": row["sparse"],
        "infer_time_ms": row["infer_time_ms"],
    })

# Plot
depth_df = pd.DataFrame(depth_metrics)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for col_idx, (metric, title) in enumerate([
    ("loss", "Test Loss"),
    ("rec_x", "Reconstruction Error"),
    ("infer_time_ms", "Inference Time (ms)"),
]):
    ax = axes[col_idx]
    ax.plot(depth_df["layers"], depth_df[metric], marker="D", linewidth=2, color="#0072B2")
    ax.set_title(title)
    ax.set_xlabel("Number of Layers")
    ax.grid(True, alpha=0.25)

fig.suptitle(f"Depth Scaling — Hybrid (seed={SEED})", fontsize=14)
plt.tight_layout(rect=[0, 0.08, 1, 0.94])
plt.savefig(str(_DEMO_DIR / "demo_depth_scaling.png"), dpi=150, bbox_inches="tight")
print(f"Saved: demo_depth_scaling.png")

---

# Summary

本デモでは Hierarchical Sparse Coding の全5ソルバを MNIST (seed=1) で学習し、以下の結果を示しました:

| セクション | 内容 |
|-----------|------|
| §2 | 各手法が学習した辞書原子（D₁, D₂, D₁·D₂）の可視化 |
| §3 | MNIST 画像の再構成結果比較 |
| §4 | テスト指標（loss, inference time）の棒グラフ |
| §5 | 訓練中の loss 遷移（学習曲線） |
| §6 | 層ごとのスパース性分析 |
| §7 | Hybrid 手法における層数スケーリング効果 |

**リポジトリの主要ファイル:**
- `HierarchicalSparseCoding.py` — コアアルゴリズム（ISTA/MFISTA/LISTA/Hybrid）
- `train_hsc.py` — 訓練ループと評価ユーティリティ
- `experiment_plotters.py` — 実験 E1–E8 のプロット関数
- `run_experiments.sh` — 全実験を起動するシェル（パラメータスイープ対応）
- `datasets.py` — データセットローダ（MNIST, Fashion-MNIST, CIFAR-10, BSDS500 など）

フル実験の再現:
```bash
./run_experiments.sh e1  # Core comparison
./run_experiments.sh e2  # Quality-latency sweep
./run_experiments.sh e3  # Depth scaling
# ... etc.
```